# Save and load the eliobj

This guide shows how to save an `eliobj` to disk and load it again.
It works before and after the fit.

In [1]:
import os
import shutil

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# remove the files of an earlier run
shutil.rmtree("results", ignore_errors=True)

## Imports

In [2]:
from typing import Any

import tensorflow_probability as tfp

import elicito as el
from elicito.fit import Fit

tfd = tfp.distributions

## Set up the eliobj

Any `eliobj` works.
This guide uses a normal outcome with an unknown mean.

In [3]:
class Model:
    """Normal outcome with an unknown mean"""

    def __call__(self, prior_samples: Any) -> dict[str, Any]:
        """
        Simulate the outcome

        Parameters
        ----------
        prior_samples
            samples from the prior distributions

        Returns
        -------
        :
            dictionary with the model outputs
        """
        ypred = tfd.Normal(loc=prior_samples[:, :, 0], scale=1.0).sample()
        return dict(ypred=ypred)


parameters = [
    el.parameter(
        name="mu",
        family=tfd.Normal,
        hyperparams=dict(loc=el.hyper("mu0"), scale=el.hyper("sigma0", lower=0)),
    )
]
targets = [el.quantiles("ypred", probs=(0.25, 0.50, 0.75))]
expert = {"ypred": [6.5, 8.0, 9.5]}

eliobj = el.Elicit(Model(), targets, parameters, epochs=100, verbose=False)

## Save the eliobj

Give a `name` or a `file`, not both.

- `name="m1"` saves to `./results/{method}/{name}_{seed}.pkl`.
  `method` is `parametric_prior` or `deep_prior`.
  `seed` is the first of the `seeds`.
- `file="results/m1"` saves to `results/m1.pkl`.

In [4]:
eliobj.save(name="m1")
eliobj.save(file="results/m1")

saved in: ./results/parametric_prior/m1_0.pkl
saved in: ./results/m1.pkl


## Load and fit the eliobj

`el.Elicit.load` returns the saved `eliobj`.
An unfitted `eliobj` needs the answers of the expert to fit.

In [5]:
eliobj_m1 = el.Elicit.load("results/m1.pkl")
fit = eliobj_m1.fit(expert)
fit

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name 'Any' is not defined
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Method: parametric_prior (100 epochs)
Shown: replication [0] (best)

Hyperparameters:
         estimate
mu0          7.86
sigma0       1.67

Priors:
         mean         sd        q05        q95
mu       7.85       1.68       5.08       10.6

Fit to expert:
             expert   elicited      error
ypred[0]        6.5       6.55     0.0517
ypred[1]          8       7.84      -0.16
ypred[2]        9.5       9.15     -0.347

Loss by target:
       technique       loss
ypred  quantiles      0.249

Final loss:
   final_loss
0       0.249

## Save and load a fitted eliobj

A fitted `eliobj` keeps its results and the answers of the expert.
To inspect the loaded results, give the `eliobj` to `Fit`.

In [6]:
eliobj_m1.save(name="m2")

eliobj_m2 = el.Elicit.load("results/parametric_prior/m2_0.pkl")
Fit(eliobj_m2)

saved in: ./results/parametric_prior/m2_0.pkl


Method: parametric_prior (100 epochs)
Shown: replication [0] (best)

Hyperparameters:
         estimate
mu0          7.86
sigma0       1.67

Priors:
         mean         sd        q05        q95
mu       7.85       1.68       5.08       10.6

Fit to expert:
             expert   elicited      error
ypred[0]        6.5       6.55     0.0517
ypred[1]          8       7.84      -0.16
ypred[2]        9.5       9.15     -0.347

Loss by target:
       technique       loss
ypred  quantiles      0.249

Final loss:
   final_loss
0       0.249

## Fit a fitted eliobj again

`fit()` raises a `ValueError` for a fitted `eliobj`.
To replace the results, set `overwrite=True`.

In [7]:
fit = eliobj_m2.fit(expert, overwrite=True)

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name 'Any' is not defined
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


## Replace a saved file

`save()` raises a `FileExistsError` if the file exists.
To replace the file, set `overwrite=True`.

In [8]:
eliobj_m2.save(name="m1", overwrite=True)

saved in: ./results/parametric_prior/m1_0.pkl
